# E16：简单回归与可复现分析报告

**课程**：Python 程序设计  
**适用对象**：应用统计学专业一年级  
**实验类型**：综合性实验  
**建议学时**：2 学时

## 实验目标

完成本实验后，你应当能够：

1. 识别简单回归中的响应变量和解释变量。
2. 使用 Statsmodels 公式接口拟合简单 OLS。
3. 提取并解释截距、斜率、拟合值、残差和 \(R^2\)。
4. 使用散点图、拟合线和基础残差图检查结果。
5. 复用 E15 结果，形成包含问题、数据、EDA、模型、结论和局限的可复现报告。

## 实验信息

- 姓名：  
- 学号：  
- 班级：  
- 日期：

## 提交要求

1. 在本 Notebook 中完成全部任务，不删除题目、提示和课程提供的数据单元。
2. 保留最终可运行的代码、统计结果、图形和必要的 Markdown 说明。
3. 不修改或覆盖 `lab/shared` 中的官方分析视图与检查点。
4. 提交前重启内核并按顺序运行全部单元，核对变量角色、统计口径和结果方向。
5. 如使用帮助文档、代码补全或 AI 工具，应在实验小结中说明用途和核验方式。

## Jupyter 使用提示

- 先运行课程提供的路径和输入准备单元，再完成后续任务。
- 修改代码后必须重新运行当前单元，后续结果才会更新。
- 统计结果异常时，先检查分组方向、变量名、缺失值和数据类型。
- 如果变量状态混乱，重启内核并从上到下重新运行。

## 建议节奏

- 输入与 E15 前序结果核验：10 分钟
- 散点图与变量角色：10 分钟
- 简单 OLS 拟合与核心结果：15 分钟
- 拟合线：10 分钟
- 残差表和残差图：15 分钟
- 结果导出：10 分钟
- 综合报告、核验与提交：20 分钟

## 最终主线：复用前序成果，不重新完成前五章

学生表继续采用 E12 个人成果优先、官方分析视图兜底。两组比较采用 E15 个人结果优先、官方 E15 检查点兜底。E16 只新增简单回归，并把已有证据组织成最终报告；不要求重新清洗数据、重画第五章全部图形或重做 E15 检验。

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import statsmodels.formula.api as smf

sns.set_theme(style="whitegrid")

# 有可靠个人成果时保持 True；需要从官方结果开始时改为 False。
USE_STUDENT_E12_OUTPUT = True
USE_STUDENT_E15_OUTPUT = True

chapter6_dir = Path(".").resolve()
notebook_names = ["e16.ipynb", "e16_inclass.ipynb"]
if not any((chapter6_dir / name).exists() for name in notebook_names):
    chapter6_dir = Path("lab/applied-stat-64/chapter6").resolve()

shared_dir = chapter6_dir.parents[1] / "shared"
student_e12_path = (
    chapter6_dir.parent
    / "chapter4"
    / "e12_output"
    / "e12_analysis_ready.csv"
)
student_e15_path = (
    chapter6_dir / "e15_output" / "e15_group_comparison.csv"
)
official_analysis_path = (
    shared_dir / "analysis" / "student_analysis.csv"
)
official_e15_path = (
    shared_dir / "analysis" / "e15_group_comparison.csv"
)
output_dir = chapter6_dir / "e16_output"

analysis_columns = [
    "student_id",
    "support_plan",
    "baseline_score",
    "study_hours",
    "attendance_rate",
    "final_score",
]
e15_columns = [
    "outcome",
    "group_1",
    "group_2",
    "n_group_1",
    "n_group_2",
    "mean_group_1",
    "mean_group_2",
    "mean_difference",
    "ci_low",
    "ci_high",
    "t_statistic",
    "p_value",
]

if USE_STUDENT_E12_OUTPUT and student_e12_path.exists():
    raw_analysis = pd.read_csv(student_e12_path)
    missing = [
        column for column in analysis_columns
        if column not in raw_analysis.columns
    ]
    if missing:
        raise ValueError(
            f"E12 个人成果缺少字段：{missing}。"
            "请修正个人成果，或关闭对应输入开关。"
        )
    analysis_df = raw_analysis[analysis_columns].copy()
    print("学生表来源：E12 个人成果")
else:
    if USE_STUDENT_E12_OUTPUT:
        print("未找到 E12 个人成果，改用官方分析视图。")
    analysis_df = pd.read_csv(official_analysis_path)
    print("学生表来源：官方分析视图")

if USE_STUDENT_E15_OUTPUT and student_e15_path.exists():
    e15_result = pd.read_csv(student_e15_path)
    print("两组比较来源：E15 个人成果")
else:
    if USE_STUDENT_E15_OUTPUT:
        print("未找到 E15 个人成果，改用官方 E15 检查点。")
    e15_result = pd.read_csv(official_e15_path)
    print("两组比较来源：官方 E15 检查点")

analysis_checks = {
    "学生表字段正确": analysis_df.columns.tolist() == analysis_columns,
    "学生表为 160 行": len(analysis_df) == 160,
    "student_id 唯一": analysis_df["student_id"].is_unique,
    "学生表无缺失": not analysis_df.isna().any().any(),
}
e15_checks = {
    "E15 字段正确": e15_result.columns.tolist() == e15_columns,
    "E15 只有一行": len(e15_result) == 1,
    "E15 结果变量正确": e15_result.loc[0, "outcome"] == "final_score",
    "E15 组序正确": (
        e15_result.loc[0, "group_1"] == "feedback"
        and e15_result.loc[0, "group_2"] == "regular"
    ),
    "E15 方向一致": (
        (e15_result.loc[0, "mean_difference"] > 0)
        == (e15_result.loc[0, "t_statistic"] > 0)
    ),
}
input_checks = {**analysis_checks, **e15_checks}
for check_name, passed in input_checks.items():
    print(check_name, "：", passed)
if not all(input_checks.values()):
    failed = [name for name, passed in input_checks.items() if not passed]
    raise ValueError(
        "输入核验未通过：" + "、".join(failed)
        + "。请修正个人成果，或关闭对应输入开关。"
    )

print("学生表形状：", analysis_df.shape)
print("E15 结果形状：", e15_result.shape)

In [ ]:
# 课程准备：确认 E15 结果确实对应当前学生表。
if set(analysis_df["support_plan"]) != {"feedback", "regular"}:
    raise ValueError("当前学生表的支持方案取值不正确。")

e15_numeric_columns = [
    "n_group_1",
    "n_group_2",
    "mean_group_1",
    "mean_group_2",
    "mean_difference",
    "ci_low",
    "ci_high",
    "t_statistic",
    "p_value",
]
e15_numeric = e15_result[e15_numeric_columns].apply(
    pd.to_numeric,
    errors="coerce",
)
if e15_numeric.isna().any().any():
    raise ValueError(
        "E15 数值字段包含无效值。"
        "请修正个人成果，或关闭对应输入开关。"
    )

result_row = e15_numeric.iloc[0]
expected_group_stats = (
    analysis_df.groupby("support_plan")["final_score"]
    .agg(["count", "mean"])
)
tolerance = 1e-5

def close_enough(left, right):
    return abs(float(left) - float(right)) <= tolerance

mean_difference = result_row["mean_difference"]
t_statistic = result_row["t_statistic"]
e15_content_checks = {
    "E15 样本量与学生表一致": (
        close_enough(
            result_row["n_group_1"],
            expected_group_stats.loc["feedback", "count"],
        )
        and close_enough(
            result_row["n_group_2"],
            expected_group_stats.loc["regular", "count"],
        )
    ),
    "E15 两组均值与学生表一致": (
        close_enough(
            result_row["mean_group_1"],
            expected_group_stats.loc["feedback", "mean"],
        )
        and close_enough(
            result_row["mean_group_2"],
            expected_group_stats.loc["regular", "mean"],
        )
    ),
    "E15 均值差内部一致": close_enough(
        mean_difference,
        result_row["mean_group_1"] - result_row["mean_group_2"],
    ),
    "E15 置信区间顺序正确": (
        result_row["ci_low"] <= result_row["ci_high"]
    ),
    "E15 置信区间包含均值差": (
        result_row["ci_low"]
        <= mean_difference
        <= result_row["ci_high"]
    ),
    "E15 p 值范围正确": 0 <= result_row["p_value"] <= 1,
    "E15 方向一致": (
        close_enough(mean_difference, 0)
        and close_enough(t_statistic, 0)
    ) or (mean_difference * t_statistic > 0),
}
for check_name, passed in e15_content_checks.items():
    print(check_name, "：", passed)
if not all(e15_content_checks.values()):
    failed = [
        name for name, passed in e15_content_checks.items()
        if not passed
    ]
    raise ValueError(
        "E15 结果内容核验未通过：" + "、".join(failed)
        + "。请修正个人成果，或关闭对应输入开关。"
    )

## 任务 1：识别变量角色并绘制散点图

研究问题：当前模拟样本中，每周学习时长与期末 Python 成绩之间有什么线性关系？

- 响应变量：`final_score`
- 解释变量：`study_hours`
- 观测单位：学生

检查缺失和范围后绘制散点图。描述关系方向、离散程度和可能的极端记录，不把散点图解释成因果证据。

In [ ]:
print(
    analysis_df[
        [...]
    ].describe().round(2)
)
print(
    analysis_df[
        [...]
    ].isna().sum()
)

scatter_fig, ax = plt.subplots(figsize=(7, 4))
sns.scatterplot(
    data=analysis_df,
    x=...,
    y=...,
    alpha=0.70,
    ax=ax,
)
ax.set(
    title=...,
    xlabel=...,
    ylabel=...,
)
scatter_fig.tight_layout()
plt.show()

## 任务 2：拟合简单 OLS 并提取核心结果

使用公式接口拟合：

```python
final_score ~ study_hours
```

只提取本课程要求的截距、斜率和 \(R^2\)，不提交未经解释的完整 `model.summary()`。斜率描述当前样本中的平均线性关联，不是每名学生的确定变化。

In [ ]:
model = smf.ols(
    ...,
    data=analysis_df,
).fit()

intercept = model.params[...]
slope = model.params[...]
r_squared = ...

print("截距：", round(intercept, 4))
print("学习时长斜率：", round(slope, 4))
print("R²：", round(r_squared, 4))

## 任务 3：把拟合线画回散点图

把模型拟合值加入数据表，按学习时长排序后绘制散点和拟合线。拟合线概括平均线性趋势，不要求所有学生都落在线上，也不支持在样本学习时长范围外随意外推。

In [ ]:
model_df = analysis_df.assign(
    fitted_score=...,
    residual=...,
)
line_df = model_df.sort_values(...)

fit_fig, ax = plt.subplots(figsize=(7, 4))
sns.scatterplot(
    data=model_df,
    x=...,
    y=...,
    alpha=0.55,
    ax=ax,
)
ax.plot(
    line_df[...],
    line_df[...],
    color="#E15759",
    linewidth=2,
)
ax.set(
    title=...,
    xlabel=...,
    ylabel=...,
)
fit_fig.tight_layout()
plt.show()

## 任务 4：构造残差并绘制基础残差图

残差等于实际成绩减去拟合成绩。绘制“拟合值—残差”散点图并添加水平零线，只进行入门级检查：

- 残差是否大致围绕 0；
- 是否存在较大的正、负残差；
- 是否出现非常明显的结构。

本实验不展开正式正态性、同方差性检验或异常点诊断。

In [ ]:
print(
    model_df[
        [...]
    ].head().round(2)
)

residual_fig, ax = plt.subplots(figsize=(7, 4))
sns.scatterplot(
    data=model_df,
    x=...,
    y=...,
    alpha=0.65,
    ax=ax,
)
ax.axhline(0, color="#E15759", linestyle="--")
ax.set(
    title=...,
    xlabel=...,
    ylabel=...,
)
residual_fig.tight_layout()
plt.show()

## 任务 5：导出回归结果和代表性图形

把样本量、截距、学习时长斜率和 \(R^2\) 整理成一行结果表，并导出拟合图与残差图。E15 结果已经作为输入读取，不在这里重新计算。

In [ ]:
regression_result = pd.DataFrame(
    [
        {
            "outcome": "final_score",
            "predictor": "study_hours",
            "n": ...,
            "intercept": ...,
            "slope": ...,
            "r_squared": ...,
        }
    ]
).round(6)

output_dir.mkdir(parents=True, exist_ok=True)
result_path = output_dir / "e16_regression_results.csv"
fit_path = output_dir / "e16_ols_fit.png"
residual_path = output_dir / "e16_residuals.png"

regression_result.to_csv(result_path, index=False)
fit_fig.savefig(fit_path, dpi=150, bbox_inches="tight")
residual_fig.savefig(
    residual_path,
    dpi=150,
    bbox_inches="tight",
)

final_checks = {
    "回归结果已导出": result_path.exists(),
    "拟合图已导出": fit_path.exists(),
    "残差图已导出": residual_path.exists(),
    "样本量为 160": regression_result.loc[0, "n"] == 160,
    "变量角色正确": (
        regression_result.loc[0, "outcome"] == "final_score"
        and regression_result.loc[0, "predictor"] == "study_hours"
    ),
    "R² 范围正确": 0 <= r_squared <= 1,
    "残差和接近 0": abs(model.resid.sum()) < 1e-8,
}
for check_name, passed in final_checks.items():
    print(check_name, "：", passed)
if not all(final_checks.values()):
    raise ValueError("最终核验未通过。")

print(regression_result.T)
print("E15 前序结果：")
print(e15_result.T)

## 任务 6：完成最终可复现分析报告

按以下结构完成报告：

1. **研究问题**：两组支持方案比较，以及学习时长与成绩的线性关系。
2. **数据来源与过程**：说明数据如何从 Chapter 4 分析就绪表进入 Chapter 5 EDA 和 Chapter 6 分析。
3. **EDA 证据**：概括分组图、散点图、拟合图和残差图。
4. **两组比较**：直接读取 E15 结果，报告均值差、区间和 p 值。
5. **简单回归**：报告斜率、\(R^2\) 和残差观察。
6. **结论与局限**：区分样本关联、统计推断和因果解释。
7. **工具核验**：说明帮助文档、代码补全或 AI 工具的用途及核验方式。

### 最终可复现分析报告

**研究问题与数据过程**

...

**EDA 与两组比较**

...

**简单回归**

...

**结论与局限**

...

**工具核验**

...

## 任务 7：课程收束检查

提交前确认：

1. Notebook 能够从头到尾运行；
2. E15 与 E16 的结果方向和变量角色一致；
3. 没有把 p 值解释为假设概率；
4. 没有把回归斜率解释为每名学生的确定变化；
5. 没有把 \(R^2\) 解释为准确率；
6. 报告说明了数据来源、处理过程、结果、局限和工具核验。

### 课程收束记录

- ...
- ...
- ...
- ...